## First step

In [1]:
!python unzip.py

9 archive(s) found. Starting extraction...

Skipping 'test.tsv.zip', already extracted at 'data/dataset_kaggle_liar/test.tsv/'.
Skipping 'base_valid.zip', already extracted at 'data/knowledge_based/base_valid/'.
Skipping 'ambiguous_test.zip', already extracted at 'data/knowledge_based/ambiguous_test/'.
Skipping 'base_test.zip', already extracted at 'data/knowledge_based/base_test/'.
Skipping 'ambiguous_valid.zip', already extracted at 'data/knowledge_based/ambiguous_valid/'.
Extracting 'Fake.zip' directly into 'data/fake_news_detection_UoVictoria/'...
Success for Fake.zip!
Extracting 'True.zip' directly into 'data/fake_news_detection_UoVictoria/'...
Success for True.zip!
Skipping 'test.csv.zip', already extracted at 'data/fake_news_detection_tweeter/test.csv/'.
Skipping 'train.csv.zip', already extracted at 'data/fake_news_detection_tweeter/train.csv/'.

All zip files are decompressed.


## Style based detection

### 1. Data preparation

If it has already been run, there is no need to run it again (but you can if you have 40 spare minutes)

In [6]:
%cd data/
!python data_extraction.py
%cd ..

%cd style_branch
!python feature_extraction.py

!python print_features.py
%cd ..

/Users/ethan/Documents/GitHub/Detection_fake_news/data

DONE! Fusion of dataset have been saved to: dataset.csv

/Users/ethan/Documents/GitHub/Detection_fake_news
/Users/ethan/Documents/GitHub/Detection_fake_news/style_branch
Loading data from ../data/dataset.csv...

Initializing NLP engine (spaCy + TextBlob/VADER)...
Loading linguistic models...
Extracting Style Metrics: 100%|███████████| 61673/61673 [42:34<00:00, 24.14it/s]

DONE! Features have been saved to: ../data/complete_train.csv
NEWS ANALYSIS:

Raw text:
ALERT! You must absolutely read this: https://www.google.com. The government is hiding 10,000 terrible and monstrous things! Wake up immediately @user! #news 😡

Loading linguistic models...
Normalized text:
ALERT! You must absolutely read this: [URL] The government is hiding [NB] terrible and monstrous things! Wake up immediately [MENTION]! #news 😡

Style vector (FEATURES):

has_hashtags                   : True
has_mentions                   : True
has_urls                   

### 2. Fine tuning RoBERTa
Same as the previous cell, you don't need to run it, it can be very very very long because of your GPU (if you have it)
(MacBook Pro M4 with 20 GPU core (48 Go unified memory) -> 41 minutes)

In [1]:
%cd style_branch
!python split_data.py

!python fine_tunning.py

%run test_fine_tuned.py
%cd ..

/Users/ethan/Documents/GitHub/Detection_fake_news/style_branch
Loading the complete dataset (Style + Text)...

Data distribution:
Block A (RoBERTa Train) : 37003 rows
Block B (Training) : 12335 rows
Block C (Final Test)    : 12335 rows

Sanity Check - Class Distribution (Fake=1, True=0):
Block A: 
label
0    0.5
1    0.5
Name: proportion, dtype: float64
Block B: 
label
0    0.5
1    0.5
Name: proportion, dtype: float64
Block C: 
label
0    0.5
1    0.5
Name: proportion, dtype: float64

Splitting complete and files saved!
Apple Silicon (MPS) detected! Training will be hardware-accelerated.
Loading CSV files...
Word tokenization in progress... This may take a minute.
Loading weights: 100%|██████████████████████| 101/101 [00:00<00:00, 8093.55it/s]
RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


Analyzed sentence: 'The Federal Reserve announced a 0.5% increase in interest rates today.'
TRUE NEWS probability: 78.63%
FAKE NEWS probability: 21.37%

Analyzed sentence: 'BREAKING: Alien spaceship found hidden under the White House! The government is lying to us!!!'
TRUE NEWS probability: 6.42%
FAKE NEWS probability: 93.58%
/Users/ethan/Documents/GitHub/Detection_fake_news


### 3. Random Forest x XGBoost competition
If warnings appears about killing child process it is because you are on MacOs and it auto kill child process and the Python processus find any child process already kild by native MacOs processsus so it's just a warning and no action required. Others os aren't concerned.

In [2]:
%cd style_branch
!python model_comp.py
%cd ..

/Users/ethan/Documents/GitHub/Detection_fake_news/style_branch
Loading data (Style + RoBERTa Semantics)...

Configuration: 15 combinations tested per model (Cross-Validation x5).
Optimizing Random Forest...

Training Random Forest: 100%|███████████████████| 75/75 [00:19<00:00,  3.94it/s]
Finished in 0.32 min. Best parameters: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_depth': 10, 'class_weight': 'balanced'}
Optimizing XGBoost...

Training XGBoost: 100%|█████████████████████████| 75/75 [00:02<00:00, 26.10it/s]
Finished in 0.05 min. Best parameters: {'subsample': 0.8, 'n_estimators': 500, 'max_depth': 3, 'learning_rate': 0.01, 'colsample_bytree': 0.9}

Model                | Accuracy             | ROC-AUC (Quality)    | F1 Score             | Log Loss             |
------------------------------------------------------------------------------------------------------------------
Random Forest        |               92.24% |               98.44% |           

In [4]:
# it's better to use in terminal

# Tests
# BREAKING NEWS!!! 🚨 Government officials are secretly hiding ALIEN technology under the White House! The mainstream media is LYING to you!!! WAKE UP! #Truth #Conspiracy
# WASHINGTON (Reuters) - The Federal Reserve announced a 0.5% increase in interest rates on Tuesday, aiming to stabilize inflation across the country.
%cd style_branch
%run inference_pipeline.py

[Errno 2] No such file or directory: 'style_branch'
/Users/ethan/Documents/GitHub/Detection_fake_news/style_branch
Initializing Detector (Loading into RAM...)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading linguistic models...
Detector is ready and loaded in memory!

 Type 'q' to quit.
